# OFDM PHY receiver pipeline (Steps 1-5)
LS → delay-domain filtering → ICI/BEM → banded LDLᴴ equalizer.

Run cells top to bottom. Definitions first, then one experiment cell per step.

**Conventions:** unitary FFTs, SNR = 1/σ² with unit-power symbols and channel, model `Y = Hf X + n` (Hf diagonal when static, banded when there's Doppler).

In [ ]:
"""
OFDM PHY receiver pipeline, Steps 1-5
  Step 1: modem + IFFT/FFT + CP + multipath/Doppler/AWGN channel
  Step 2: comb pilots, LS estimate, linear/spline interpolation, 1-tap ZF
  Step 3: delay-domain noise filtering (IDFT -> window -> DFT)
  Step 4: high Doppler -> ICI, CE-BEM channel estimation (pilot clusters)
  Step 5: banded MMSE equalizer via LDL^H, O(N Q^2) instead of O(N^3)

Conventions
  * FFTs are unitary (norm="ortho"), so noise variance is the same in time and
    frequency, and the DFT matrix F/sqrt(N) satisfies F^H F = I.
  * SNR = 1/sigma^2 with unit-power symbols and unit-power channel.
  * Frequency-domain model: Y = Hf X + n, where Hf is N x N.
    Static channel -> Hf diagonal. Doppler -> Hf gets off-diagonals (ICI).
"""
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

rng = np.random.default_rng(1)

# ----------------------------------------------------------------------------
# Global parameters
# ----------------------------------------------------------------------------
N = 256            # subcarriers
CP = 32            # cyclic prefix length (samples)
BITS_PER_SYM = 2   # 2 = QPSK, 4 = 16-QAM

# Step 2/3 (static channel)
L_STATIC = 8       # channel taps
PILOT_SPACING = 8  # comb pilot spacing (N/spacing must exceed L)
WIN_LEN = 16       # delay-domain window length (>= L, <= CP)

# Step 4/5 (high Doppler)
L_MOBILE = 4       # channel taps
EPS = 0.2          # normalized Doppler f_d / subcarrier spacing
Q = 2              # BEM / band half-width (diagonals on each side)
P_CLUSTERS = 16    # number of pilot clusters
PILOT_AMP = 2.0    # pilot amplitude boost for clusters
%matplotlib inline

## Step 1: modem, OFDM, channel
Mapper/demapper, IFFT+CP / CP removal+FFT, Rayleigh multipath (static or Doppler), AWGN, and a genie `true_Hf` for analysis.

In [ ]:
class Modem:
    """Gray-mapped QPSK / 16-QAM, unit average power."""

    def __init__(self, bps):
        self.bps = bps
        M = 2 ** bps
        if bps == 2:
            lv = {0: 1.0, 1: -1.0}
            tab = [(lv[i >> 1] + 1j * lv[i & 1]) for i in range(M)]
            norm = np.sqrt(2)
        elif bps == 4:
            lv = {0b00: -3.0, 0b01: -1.0, 0b11: 1.0, 0b10: 3.0}
            tab = [(lv[i >> 2] + 1j * lv[i & 3]) for i in range(M)]
            norm = np.sqrt(10)
        else:
            raise ValueError("bps must be 2 or 4")
        self.table = np.array(tab) / norm
        self.weights = 2 ** np.arange(bps - 1, -1, -1)

    def mod(self, bits):
        idx = bits.reshape(-1, self.bps) @ self.weights
        return self.table[idx]

    def demod(self, sym):
        d = np.abs(sym[:, None] - self.table[None, :])
        idx = np.argmin(d, axis=1)
        return ((idx[:, None] >> np.arange(self.bps - 1, -1, -1)) & 1).reshape(-1)


modem = Modem(BITS_PER_SYM)


def ofdm_tx(X):
    """Frequency-domain symbols -> IFFT -> add CP."""
    x = np.fft.ifft(X, norm="ortho")
    return np.concatenate([x[-CP:], x])


def ofdm_rx(y):
    """Remove CP -> FFT."""
    return np.fft.fft(y[CP:CP + N], norm="ortho")


def make_channel(L, eps, n_samples):
    """
    Tap-delay-line Rayleigh channel, exponential power profile, unit total power.
    eps = 0  : static over the symbol (one complex Gaussian per tap)
    eps > 0  : time-varying taps via sum-of-sinusoids (Clarke/Jakes),
               max Doppler = eps subcarrier spacings.
    Returns h of shape (L, n_samples).
    """
    pdp = np.exp(-np.arange(L) / (L / 3.0))
    pdp /= pdp.sum()
    if eps == 0:
        g = (rng.standard_normal(L) + 1j * rng.standard_normal(L)) / np.sqrt(2)
        return (np.sqrt(pdp) * g)[:, None] * np.ones((1, n_samples))
    M = 32
    theta = rng.uniform(0, 2 * np.pi, (L, M))
    phi = rng.uniform(0, 2 * np.pi, (L, M))
    n = np.arange(n_samples)
    ph = 2 * np.pi * eps * np.cos(theta)[:, :, None] * n[None, None, :] / N + phi[:, :, None]
    h = np.exp(1j * ph).sum(axis=1) / np.sqrt(M)
    return np.sqrt(pdp)[:, None] * h


def apply_channel(x, h, snr_db):
    """y[n] = sum_l h_l[n] x[n-l] + AWGN (works for time-varying h)."""
    y = np.zeros(len(x), dtype=complex)
    for l in range(h.shape[0]):
        y[l:] += h[l, l:] * x[:len(x) - l]
    sigma2 = 10 ** (-snr_db / 10)
    w = np.sqrt(sigma2 / 2) * (rng.standard_normal(len(x)) + 1j * rng.standard_normal(len(x)))
    return y + w, sigma2


def true_Hf(h):
    """Genie N x N frequency-domain channel matrix: Hf = F Hc F^H."""
    Hc = np.zeros((N, N), dtype=complex)
    n = np.arange(N)
    for l in range(h.shape[0]):
        Hc[n, (n - l) % N] = h[l, CP:CP + N]
    F = np.fft.fft(np.eye(N), axis=0) / np.sqrt(N)
    return F @ Hc @ F.conj().T

## Step 2: comb pilots, LS, interpolation, ZF
`H_LS = Y_p / X_p` at pilots, interpolate to all subcarriers (linear or cubic spline).

In [ ]:
PILOT_IDX = np.unique(np.r_[np.arange(0, N, PILOT_SPACING), N - 1])
DATA_IDX_COMB = np.setdiff1d(np.arange(N), PILOT_IDX)
PILOT_VAL = (1 + 1j) / np.sqrt(2)


def ls_estimate(Y, kind):
    """H_LS at pilots = Y_p / X_p, then interpolate to every subcarrier."""
    Hp = Y[PILOT_IDX] / PILOT_VAL
    k = np.arange(N)
    fr = interp1d(PILOT_IDX, Hp.real, kind=kind)(k)
    fi = interp1d(PILOT_IDX, Hp.imag, kind=kind)(k)
    return fr + 1j * fi

## Step 3: delay-domain noise filtering
IDFT → keep first `WIN_LEN` taps → DFT. `projection_matrix` gives the same filter as one matrix P = F D Fᴴ / N.

In [ ]:
def delay_filter(H_ls, win_len=WIN_LEN):
    """H_clean = F D (1/N) F^H H_LS   (IDFT -> keep first win_len taps -> DFT)"""
    h = np.fft.ifft(H_ls)       # numpy ifft already includes 1/N
    h[win_len:] = 0
    return np.fft.fft(h)


def projection_matrix(win_len=WIN_LEN):
    """The same filter as one N x N matrix P (for inspection; not used in the sim)."""
    F = np.fft.fft(np.eye(N), axis=0)
    D = np.diag((np.arange(N) < win_len).astype(float))
    return F @ D @ F.conj().T / N


def sim_static(snrs, n_sym):
    names = ["Perfect CSI", "LS linear", "LS spline", "LS linear + delay filt", "LS spline + delay filt"]
    errs = np.zeros((len(names), len(snrs)))
    nbits = len(DATA_IDX_COMB) * BITS_PER_SYM * n_sym
    mse = np.zeros((len(names) - 1, len(snrs)))
    for _ in range(n_sym):
        bits = rng.integers(0, 2, len(DATA_IDX_COMB) * BITS_PER_SYM)
        X = np.zeros(N, dtype=complex)
        X[DATA_IDX_COMB] = modem.mod(bits)
        X[PILOT_IDX] = PILOT_VAL
        x = ofdm_tx(X)
        h = make_channel(L_STATIC, 0.0, N + CP)
        H_true = np.fft.fft(h[:, 0], N) * 1.0  # unitary DFT of taps (sum, no 1/sqrt(N))
        for si, snr in enumerate(snrs):
            y, _ = apply_channel(x, h, snr)
            Y = ofdm_rx(y)
            Hl = ls_estimate(Y, "linear")
            Hs = ls_estimate(Y, "cubic")
            ests = [H_true, Hl, Hs, delay_filter(Hl), delay_filter(Hs)]
            for mi, He in enumerate(ests):
                Xh = Y[DATA_IDX_COMB] / He[DATA_IDX_COMB]            # 1-tap ZF
                errs[mi, si] += np.sum(modem.demod(Xh) != bits)
                if mi > 0:
                    mse[mi - 1, si] += np.mean(np.abs(He - H_true) ** 2)
    return names, errs / nbits, mse / n_sym

## Step 4: high Doppler, ICI, CE-BEM
With integer Doppler bins, `Hf[m+q, m] = D_q[m]` is exactly banded, and each diagonal D_q is an L-tap frequency response. Pilot clusters (pilot + 2Q zeros each side) let us read D_q at the cluster centres, then LS-fit L taps per q.

In [ ]:
def offdiag_energy_fraction(Hf):
    tot = np.sum(np.abs(Hf) ** 2)
    return 1 - np.sum(np.abs(np.diag(Hf)) ** 2) / tot


def bem_fit_nmse(h, Qb):
    """How well do 2Qb+1 complex exponentials describe the true taps? (genie check)"""
    n = np.arange(N)
    B = np.exp(2j * np.pi * np.outer(n, np.arange(-Qb, Qb + 1)) / N)   # N x (2Qb+1), orthogonal
    ht = h[:, CP:CP + N]
    c = ht @ B.conj() / N
    rec = c @ B.T
    return np.sum(np.abs(ht - rec) ** 2) / np.sum(np.abs(ht) ** 2)


# Pilot clusters: [2Q zeros | pilot | 2Q zeros]. The guard stops data ICI from
# contaminating the received bins we use to read off the BEM coefficients.
CENTERS = ((np.arange(P_CLUSTERS) + 0.5) * N / P_CLUSTERS).astype(int)
NULL_BINS = np.unique(np.concatenate([CENTERS + j for j in range(-2 * Q, 2 * Q + 1)]))
DATA_IDX_MOB = np.setdiff1d(np.arange(N), NULL_BINS)


def bem_estimate(Y, L_est=L_MOBILE):
    """
    CE-BEM with integer Doppler bins q = -Q..Q makes Hf exactly banded:
        Hf[m+q, m] = D_q[m],   D_q[m] = sum_l c_{l,q} e^{-j2 pi m l / N}
    i.e. every diagonal is a normal frequency response of an L-tap channel
    (this is the Step-3 idea applied to each diagonal).
    Read D_q at pilot centres:  Y[kp+q] = D_q[kp] * X_p, then LS-fit L taps per q.
    Returns D with shape (2Q+1, N); D[Q] is the main diagonal.
    """
    A = np.exp(-2j * np.pi * np.outer(CENTERS, np.arange(L_est)) / N)       # Pc x L
    Apinv = np.linalg.pinv(A)
    Fl = np.exp(-2j * np.pi * np.outer(np.arange(N), np.arange(L_est)) / N)  # N x L
    D = np.zeros((2 * Q + 1, N), dtype=complex)
    for qi, q in enumerate(range(-Q, Q + 1)):
        d = Y[(CENTERS + q) % N] / (PILOT_AMP * PILOT_VAL)
        D[qi] = Fl @ (Apinv @ d)
    return D


def band_from_D(D):
    """Zero entries that would fall outside the matrix (no circular corners)."""
    Hb = D.copy()
    m = np.arange(N)
    for qi, q in enumerate(range(-Q, Q + 1)):
        Hb[qi, (m + q < 0) | (m + q >= N)] = 0
    return Hb


def dense_from_band(Hb):
    Hd = np.zeros((N, N), dtype=complex)
    m = np.arange(N)
    for qi, q in enumerate(range(-Q, Q + 1)):
        ok = (m + q >= 0) & (m + q < N)
        Hd[m[ok] + q, m[ok]] = Hb[qi, ok]
    return Hd

## Step 5: equalizers
Dense MMSE O(N³) vs banded LDLᴴ O(N Q²). Same answer, since the only approximation is banding H.

In [ ]:
def mmse_dense(H, y, sigma2):
    """Brute force, O(N^3):  x = (H^H H + s2 I)^-1 H^H y"""
    A = H.conj().T @ H + sigma2 * np.eye(N)
    return np.linalg.solve(A, H.conj().T @ y)


def mmse_banded_ldl(Hb, y, sigma2):
    """
    Same MMSE solution, using the band structure.
      H has half-bandwidth Q  ->  A = H^H H + s2 I has half-bandwidth w = 2Q
      A = L D L^H with L lower-banded (no fill-in).
    Cost: factor O(N w^2), two substitutions O(N w).
    """
    w = 2 * Q
    # A's upper diagonals: Aup[d, i] = A[i, i+d]
    Aup = np.zeros((w + 1, N), dtype=complex)
    for d in range(w + 1):
        acc = np.zeros(N - d, dtype=complex)
        for p in range(d - Q, Q + 1):
            acc += np.conj(Hb[p + Q, :N - d]) * Hb[p - d + Q, d:]
        Aup[d, :N - d] = acc
    Aup[0] += sigma2

    # b = H^H y  (banded mat-vec)
    ypad = np.concatenate([np.zeros(Q), y, np.zeros(Q)])
    b = np.zeros(N, dtype=complex)
    for p in range(-Q, Q + 1):
        b += np.conj(Hb[p + Q]) * ypad[Q + p: Q + p + N]

    # LDL^H.  Lb[i, t] = L[i, i-t]  (t = 1..w), dg = diag(D)
    Lb = np.zeros((N, w + 1), dtype=complex)
    dg = np.zeros(N)
    for j in range(N):
        s = Aup[0, j].real
        for t in range(1, min(w, j) + 1):
            s -= abs(Lb[j, t]) ** 2 * dg[j - t]
        dg[j] = s
        for i in range(j + 1, min(N, j + w + 1)):
            v = np.conj(Aup[i - j, j])                    # A[i, j]
            for k in range(max(0, i - w), j):
                v -= Lb[i, i - k] * dg[k] * np.conj(Lb[j, j - k])
            Lb[i, i - j] = v / dg[j]

    # forward:  L z = b
    z = b.copy()
    for i in range(N):
        for t in range(1, min(w, i) + 1):
            z[i] -= Lb[i, t] * z[i - t]
    u = z / dg                                           # D u = z
    # backward: L^H x = u
    x = u.copy()
    for i in range(N - 1, -1, -1):
        for t in range(1, min(w, N - 1 - i) + 1):
            x[i] -= np.conj(Lb[i + t, t]) * x[i + t]
    return x


def sim_genie_ici(eps_list, snrs, n_sym):
    """ICI floor: 1-tap ZF with PERFECT knowledge of the diagonal of Hf."""
    res = np.zeros((len(eps_list), len(snrs)))
    nbits = N * BITS_PER_SYM * n_sym
    for ei, eps in enumerate(eps_list):
        for _ in range(n_sym):
            bits = rng.integers(0, 2, N * BITS_PER_SYM)
            x = ofdm_tx(modem.mod(bits))
            h = make_channel(L_MOBILE, eps, N + CP)
            Hd = np.diag(true_Hf(h))
            for si, snr in enumerate(snrs):
                y, _ = apply_channel(x, h, snr)
                Xh = ofdm_rx(y) / Hd
                res[ei, si] += np.sum(modem.demod(Xh) != bits)
    return res / nbits


def sim_mobile(snrs, n_sym):
    names = ["1-tap ZF (BEM diag)", "Full MMSE (BEM, O(N^3))", "Banded LDL^H (BEM, O(NQ^2))",
             "Genie full MMSE (true Hf)"]
    errs = np.zeros((len(names), len(snrs)))
    tim = np.zeros(len(names))
    nbits = len(DATA_IDX_MOB) * BITS_PER_SYM * n_sym
    maxdiff = 0.0
    for _ in range(n_sym):
        bits = rng.integers(0, 2, len(DATA_IDX_MOB) * BITS_PER_SYM)
        X = np.zeros(N, dtype=complex)
        X[DATA_IDX_MOB] = modem.mod(bits)
        X[CENTERS] = PILOT_AMP * PILOT_VAL
        x = ofdm_tx(X)
        h = make_channel(L_MOBILE, EPS, N + CP)
        Hf_true = true_Hf(h)
        for si, snr in enumerate(snrs):
            y, s2 = apply_channel(x, h, snr)
            Y = ofdm_rx(y)
            D = bem_estimate(Y)
            Hb = band_from_D(D)
            Hd = dense_from_band(Hb)

            t = time.perf_counter(); xh0 = Y / D[Q]; tim[0] += time.perf_counter() - t
            t = time.perf_counter(); xh1 = mmse_dense(Hd, Y, s2); tim[1] += time.perf_counter() - t
            t = time.perf_counter(); xh2 = mmse_banded_ldl(Hb, Y, s2); tim[2] += time.perf_counter() - t
            t = time.perf_counter(); xh3 = mmse_dense(Hf_true, Y, s2); tim[3] += time.perf_counter() - t
            maxdiff = max(maxdiff, np.max(np.abs(xh1 - xh2)))
            for mi, xh in enumerate([xh0, xh1, xh2, xh3]):
                errs[mi, si] += np.sum(modem.demod(xh[DATA_IDX_MOB]) != bits)
    tim /= (n_sym * len(snrs))
    return names, errs / nbits, tim, maxdiff


def complexity_benchmark(sizes=(128, 256, 512, 1024), Qb=2):
    """Time one solve: dense O(N^3) vs banded LDL^H, on a random banded H."""
    global N, Q
    N0, Q0 = N, Q
    Q = Qb
    rows = []
    for n in sizes:
        N = n
        D = rng.standard_normal((2 * Q + 1, N)) + 1j * rng.standard_normal((2 * Q + 1, N))
        Hb = band_from_D(D)
        Hd = dense_from_band(Hb)
        y = rng.standard_normal(N) + 1j * rng.standard_normal(N)
        t = time.perf_counter(); xd = mmse_dense(Hd, y, 0.1); td = time.perf_counter() - t
        t = time.perf_counter(); xb = mmse_banded_ldl(Hb, y, 0.1); tb = time.perf_counter() - t
        rows.append((n, td * 1e3, tb * 1e3, np.max(np.abs(xd - xb))))
    N, Q = N0, Q0
    return rows

---
# Experiments

### Steps 2-3: static channel, BER and channel MSE

In [ ]:
snrs = np.arange(0, 31, 5)
names, ber, mse = sim_static(snrs, n_sym=300)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for n_, b in zip(names, ber):
    ax[0].semilogy(snrs, np.maximum(b, 1e-6), marker="o", label=n_)
ax[0].set(xlabel="SNR (dB)", ylabel="BER", title="Static multipath: LS vs delay-filtered", ylim=(1e-5, 1))
ax[0].legend(); ax[0].grid(True, which="both", alpha=.3)
for n_, m in zip(names[1:], mse):
    ax[1].semilogy(snrs, m, marker="o", label=n_)
ax[1].set(xlabel="SNR (dB)", ylabel="Channel MSE", title="Channel estimation MSE")
ax[1].legend(); ax[1].grid(True, which="both", alpha=.3)
plt.tight_layout(); plt.show()

### Step 3 intuition: rows of the projection matrix P are smoothing kernels

In [ ]:
P = projection_matrix()
plt.figure(figsize=(7, 3.5))
for k in [40, 128, 200]:
    plt.plot(np.roll(np.real(P[k]), N//2 - k)[N//2-40:N//2+40], label=f"row {k}")
plt.title("Each row of P = sinc-like smoothing kernel (centered)"); plt.legend(); plt.grid(alpha=.3)
plt.show()
print("noise power kept ~ trace(P)/N =", np.real(np.trace(P))/N, " (WIN_LEN/N =", WIN_LEN/N, ")")

### Step 4: what Doppler does to Hf, and the BER floor

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
for a, eps in zip(ax, [0.0, EPS]):
    Hf = true_Hf(make_channel(L_MOBILE, eps, N + CP))
    a.imshow(20*np.log10(np.abs(Hf) + 1e-6), cmap="magma", vmin=-40, vmax=5)
    a.set_title(f"|Hf| dB, eps={eps}\noff-diag energy {offdiag_energy_fraction(Hf):.3f}")
    a.set_xlabel("input subcarrier m"); a.set_ylabel("output subcarrier k")
plt.tight_layout(); plt.show()

h = make_channel(L_MOBILE, EPS, N + CP)
print("BEM fit NMSE vs Q:", {qb: f"{bem_fit_nmse(h, qb):.1e}" for qb in range(0, 5)})

In [ ]:
eps_list = [0.0, 0.05, 0.1, 0.2]
snrs2 = np.arange(0, 41, 5)
ici = sim_genie_ici(eps_list, snrs2, n_sym=150)
plt.figure(figsize=(6.5, 4.5))
for e, b in zip(eps_list, ici):
    plt.semilogy(snrs2, np.maximum(b, 1e-6), marker="o", label=f"eps = {e}")
plt.grid(True, which="both", alpha=.3); plt.xlabel("SNR (dB)"); plt.ylabel("BER"); plt.ylim(1e-5, 1)
plt.title("ICI error floor (1-tap ZF, perfect diagonal CSI)"); plt.legend(); plt.show()

### Step 5: mobile channel, equalizer comparison (takes about a minute)

In [ ]:
snrs3 = np.arange(0, 41, 5)
names3, ber3, tim, maxdiff = sim_mobile(snrs3, n_sym=60)
plt.figure(figsize=(6.5, 4.5))
for n_, b in zip(names3, ber3):
    plt.semilogy(snrs3, np.maximum(b, 1e-6), marker="o", label=n_)
plt.grid(True, which="both", alpha=.3); plt.xlabel("SNR (dB)"); plt.ylabel("BER"); plt.ylim(1e-5, 1)
plt.title(f"High Doppler (eps={EPS}), Q={Q}"); plt.legend(fontsize=8); plt.show()
print("max |dense - banded| solution difference:", f"{maxdiff:.2e}")
print("avg ms per solve:", {n_: round(float(t)*1e3, 3) for n_, t in zip(names3, tim)})

### Complexity: dense O(N³) vs banded O(N Q²)
Banded loses at tiny N because of Python loop overhead; the crossover is around N ≈ 256.

In [ ]:
rows = complexity_benchmark()
print("      N     dense(ms)  banded(ms)   max diff")
for n, td, tb, df in rows:
    print(f"  {n:5d} {td:10.2f} {tb:10.2f}   {df:.1e}")
ns = [r[0] for r in rows]
plt.figure(figsize=(6, 4))
plt.loglog(ns, [r[1] for r in rows], "o-", label="dense  ~N^3")
plt.loglog(ns, [r[2] for r in rows], "o-", label="banded ~N")
plt.xlabel("N"); plt.ylabel("ms per solve"); plt.legend(); plt.grid(True, which="both", alpha=.3); plt.show()